In [146]:
import pandas as pd

In [147]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [148]:
df = pd.read_csv(
    "/content/drive/MyDrive/fra.txt",
    sep="\t",
    header=None,
    usecols=[0,1],
    names=["English","French"]
)

In [149]:
df

,English,French
0,Go.,Va !
1,Go.,Marche.
2,Go.,En route !
3,Go.,Bouge !
4,Hi.,Salut !
...,...,...
240516,Death is something that we're often discourage...,La mort est une chose qu'on nous décourage sou...
240517,Since there are usually multiple websites on a...,Puisqu'il y a de multiples sites web sur chaqu...
240518,If someone who doesn't know your background sa...,Si quelqu'un qui ne connaît pas vos antécédent...
240519,It may be impossible to get a completely error...,Il est peut-être impossible d'obtenir un Corpu...


In [150]:
df = df.iloc[:30000]

In [151]:
df.shape

(30000, 2)

In [152]:
df["English"] = df["English"].str.lower()
df["French"] = df["French"].str.lower()

/tmp/ipykernel_1621/425419693.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["English"] = df["English"].str.lower()
/tmp/ipykernel_1621/425419693.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["French"] = df["French"].str.lower()


In [153]:
df["English"] = df["English"].str.strip()
df["French"] = df["French"].str.strip()

/tmp/ipykernel_1621/1970090566.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["English"] = df["English"].str.strip()
/tmp/ipykernel_1621/1970090566.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["French"] = df["French"].str.strip()


In [154]:
df["French"] = "<START> " + df["French"] + " <END>"

/tmp/ipykernel_1621/3571122706.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["French"] = "<START> " + df["French"] + " <END>"


In [155]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [156]:
eng_tokenizer = Tokenizer(filters="")
eng_tokenizer.fit_on_texts(df["English"])

In [157]:
encoder_sequences = eng_tokenizer.texts_to_sequences(df["English"])

In [158]:
fra_tokenizer = Tokenizer(filters="")
fra_tokenizer.fit_on_texts(df["French"])

In [159]:
decoder_sequences = fra_tokenizer.texts_to_sequences(df["French"])

In [160]:
encoder_vocab = len(eng_tokenizer.word_index)+1
decoder_vocab = len(fra_tokenizer.word_index)+1

In [161]:
print(encoder_vocab)
print(decoder_vocab)

6367
12368


In [162]:
max_encoder_len = max(len(x) for x in encoder_sequences)

In [163]:
max_decoder_len = max(len(x) for x in decoder_sequences)

In [164]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [165]:
encoder_input_data = pad_sequences(
    encoder_sequences,
    maxlen=max_encoder_len,
    padding="post"
)

In [166]:
decoder_data = pad_sequences(
    decoder_sequences,
    maxlen=max_decoder_len,
    padding="post"
)

In [167]:
decoder_input_data = decoder_data[:,:-1]

In [168]:
decoder_target_data = decoder_data[:,1:]

In [169]:
import numpy as np

decoder_target_data = np.expand_dims(decoder_target_data,-1)

In [170]:
from tensorflow.keras.layers import Input, Embedding, LSTM

In [171]:
latent_dim = 256

In [172]:
encoder_inputs = Input(shape=(max_encoder_len,))

In [173]:
encoder_embedding = Embedding(
    encoder_vocab,
    256,
    mask_zero=True)(encoder_inputs)

In [174]:
encoder_outputs,state_h,state_c = LSTM(
    latent_dim,
    return_state=True)(encoder_embedding)

In [175]:
encoder_states = [state_h,state_c]

In [176]:
from tensorflow.keras.layers import Dense

In [177]:
decoder_inputs = Input(shape=(max_decoder_len-1,))

In [178]:
decoder_embedding_layer = Embedding(
    decoder_vocab,
    256,
    mask_zero=True
)
decoder_embedding = decoder_embedding_layer(decoder_inputs)

In [179]:
decoder_lstm = LSTM(
    latent_dim,
    return_sequences=True,
    return_state=True
)

In [180]:
decoder_outputs,_,_ = decoder_lstm(
    decoder_embedding,
    initial_state=encoder_states
)

In [181]:
decoder_dense = Dense(decoder_vocab,activation="softmax")

In [182]:
decoder_outputs = decoder_dense(decoder_outputs)

In [183]:
from tensorflow.keras.models import Model

In [184]:
model = Model(
    [encoder_inputs,decoder_inputs],
    decoder_outputs
)

In [185]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [186]:
model.summary()

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_8       │ (None, 6)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_9       │ (None, 13)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_8         │ (None, 6, 256)    │  1,629,952 │ input_layer_8[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_2         │ (None, 6)         │          0 │ input_layer_8[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_9         │ (None, 13, 256)   │  3,166,208 │ input_layer_9[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_6 (LSTM)       │ [(None, 256),     │    525,312 │ embedding_8[0][0… │
│                     │ (None, 256),      │            │ not_equal_2[0][0] │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_7 (LSTM)       │ [(None, 13, 256), │    525,312 │ embedding_9[0][0… │
│                     │ (None, 256),      │            │ lstm_6[0][1],     │
│                     │ (None, 256)]      │            │ lstm_6[0][2]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 13, 12368) │  3,178,576 │ lstm_7[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 9,025,360 (34.43 MB)

 Trainable params: 9,025,360 (34.43 MB)

 Non-trainable params: 0 (0.00 B)

In [187]:
from tensorflow.keras.callbacks import EarlyStopping

In [188]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

In [189]:
history = model.fit(
    [encoder_input_data,decoder_input_data],
    decoder_target_data,
    batch_size=64,
    epochs=50,
    validation_split=0.2,
    callbacks=[early_stop]
)

Epoch 1/50
375/375 ━━━━━━━━━━━━━━━━━━━━ 25s 60ms/step - accuracy: 0.3923 - loss: 4.5182 - val_accuracy: 0.4200 - val_loss: 4.1015
Epoch 2/50
375/375 ━━━━━━━━━━━━━━━━━━━━ 24s 64ms/step - accuracy: 0.5188 - loss: 3.2085 - val_accuracy: 0.4953 - val_loss: 3.5743
Epoch 3/50
375/375 ━━━━━━━━━━━━━━━━━━━━ 21s 57ms/step - accuracy: 0.5832 - loss: 2.6464 - val_accuracy: 0.5268 - val_loss: 3.3186
Epoch 4/50
375/375 ━━━━━━━━━━━━━━━━━━━━ 24s 64ms/step - accuracy: 0.6183 - loss: 2.2498 - val_accuracy: 0.5389 - val_loss: 3.2019
Epoch 5/50
375/375 ━━━━━━━━━━━━━━━━━━━━ 22s 59ms/step - accuracy: 0.6461 - loss: 1.9339 - val_accuracy: 0.5494 - val_loss: 3.0981
Epoch 6/50
375/375 ━━━━━━━━━━━━━━━━━━━━ 22s 57ms/step - accuracy: 0.6725 - loss: 1.6700 - val_accuracy: 0.5606 - val_loss: 3.0657
Epoch 7/50
375/375 ━━━━━━━━━━━━━━━━━━━━ 22s 58ms/step - accuracy: 0.6996 - loss: 1.4447 - val_accuracy: 0.5675 - val_loss: 3.0298
Epoch 8/50
375/375 ━━━━━━━━━━━━━━━━━━━━ 22s 58ms/step - accuracy: 0.7255 - loss: 1.2474 - 

In [190]:
encoder_model = Model(
    encoder_inputs,
    encoder_states
)

In [191]:
decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))

In [192]:
decoder_states_inputs = [
    decoder_state_input_h,
    decoder_state_input_c
]

In [193]:
decoder_embedding2 = decoder_embedding_layer(decoder_inputs)

In [194]:
decoder_outputs, state_h, state_c = decoder_lstm(
    decoder_embedding2,
    initial_state=decoder_states_inputs
)

In [195]:
decoder_outputs = decoder_dense(decoder_outputs)

In [196]:
decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs, state_h, state_c]
)

In [197]:
import numpy as np

reverse_french_index = {
    index: word
    for word, index in fra_tokenizer.word_index.items()
}

In [198]:
def decode_sequence(input_seq):

    # Encode the input sentence
    states_value = encoder_model.predict(input_seq, verbose=0)

    # Start token
    start_token = fra_tokenizer.word_index["<start>"]

    # First decoder input
    target_seq = np.array([[start_token]])

    decoded_sentence = ""

    while True:

        # Predict next word
        output_tokens, h, c = decoder_model.predict(
            [target_seq] + states_value,
            verbose=0
        )

        # Highest probability token
        sampled_token_index = np.argmax(output_tokens[0, -1, :])

        # Convert token to word
        sampled_word = reverse_french_index.get(sampled_token_index, "")

        # Stop if END token or maximum length reached
        if sampled_word == "<end>" or len(decoded_sentence.split()) >= max_decoder_len:
            break

        # Ignore START token if predicted
        if sampled_word != "<start>":
            decoded_sentence += sampled_word + " "

        # Feed predicted word back into decoder
        target_seq = np.array([[sampled_token_index]])

        # Update decoder states
        states_value = [h, c]

    return decoded_sentence.strip()

In [199]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

def translate(sentence):

    sentence = sentence.lower().strip()

    sequence = eng_tokenizer.texts_to_sequences([sentence])

    sequence = pad_sequences(
        sequence,
        maxlen=max_encoder_len,
        padding="post"
    )

    translation = decode_sequence(sequence)

    return translation

In [207]:
print(translate("how are you"))
print(translate("i love you"))
print(translate("where are you"))
print(translate("good morning"))
print(translate("thank you"))
print(translate("what is your name"))

comment allez-vous ?
je vous aime les deux.
où est ?
à tom !
je vous remercie.
comment est ton chien ?


In [208]:
#https://blog.keras.io/a-ten-minute-introduction-to-sequence-to-sequence-learning-in-keras.html

In [209]:
print(translate("Hello"))

attendez !


In [222]:
print(translate("I fell"))

je suis tombée.


In [228]:
test_sentences = [
    # Very Short
    "Go.",
    "Stop.",
    "Run.",
    "Wait.",
    "Come.",
    "Leave.",
    "Listen.",
    "Look.",
    "Help!",
    "Thanks.",
    "Hello.",
    "Goodbye.",
    "Yes.",
    "No.",
    "Sorry.",

    # Greetings
    "Hi.",
    "Good morning.",
    "Good evening.",
    "Good night.",
    "How are you?",
    "Nice to meet you.",
    "See you later.",
    "Have a nice day.",
    "Welcome.",
    "Take care.",

    # Everyday
    "I am happy.",
    "I am tired.",
    "I am hungry.",
    "I am busy.",
    "I am ready.",
    "I love you.",
    "I like coffee.",
    "I like music.",
    "I need help.",
    "I understand.",
    "I don't understand.",
    "I agree.",
    "I don't know.",

    # Questions
    "What is your name?",
    "Where are you?",
    "Where do you live?",
    "How old are you?",
    "What time is it?",
    "Who are you?",
    "Why are you here?",
    "Can you help me?",
    "Do you speak English?",
    "Are you okay?",

    # Family
    "I have a brother.",
    "I have a sister.",
    "My mother is kind.",
    "My father is a doctor.",
    "This is my family.",

    # Food
    "I like pizza.",
    "I like rice.",
    "I drink water.",
    "I want coffee.",
    "The food is delicious.",
    "The tea is hot.",

    # Travel
    "Where is the station?",
    "Where is the airport?",
    "I need a taxi.",
    "I am lost.",
    "Turn left.",
    "Turn right.",
    "Go straight.",

    # School
    "I am a student.",
    "This is my book.",
    "Open the door.",
    "Close the window.",
    "Read this.",
    "Write your name.",

    # Longer Sentences
    "I love playing football.",
    "She is reading a book.",
    "The weather is very nice today.",
    "My friend lives in Paris.",
    "I want to learn French.",
    "The train is arriving soon.",
    "Please help me find the hospital.",
    "I forgot my wallet at home.",
    "Can you repeat that again?",
    "This is a beautiful city."
]

for sentence in test_sentences:
    print(f"English   : {sentence}")
    print(f"French    : {translate(sentence)}")
    print("-" * 60)

English   : Go.
French    : !
------------------------------------------------------------
English   : Stop.
French    : on vous prie.
------------------------------------------------------------
English   : Run.
French    : fuyez !
------------------------------------------------------------
English   : Wait.
French    : attendez !
------------------------------------------------------------
English   : Come.
French    : saute.
------------------------------------------------------------
English   : Leave.
French    : je vous prie.
------------------------------------------------------------
English   : Listen.
French    : attendez !
------------------------------------------------------------
English   : Look.
French    : merci !
------------------------------------------------------------
English   : Help!
French    : à la maison !
------------------------------------------------------------
English   : Thanks.
French    : merci !
----------------------------------------------------